In [1]:
import sys

sys.path.append("..")

import warnings

import pandas as pd
import torch

import os

from distilrl.utils.misc import seed_everything
from distilrl.constants import CONFIG_DIR, DATASET_DIR, MODEL_DIR
from pathlib import Path

from datasets import Dataset

from hydra.utils import instantiate, get_class

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2
%cd ..

device = "cuda" if torch.cuda.is_available() else "cpu"
device

/home/ivanov.dko/projects/distilrl


'cpu'

Теперь посмотрим на то, как влияют на трансформер различные гиперпараметры

Прежде всего хочу сказать немного о тех метриках, что у нас есть:
1. Accuracy. Здесь всё понятно - доля правильно угаданных действий. Метрика с одной стороны хорошая, с другой - полностью нерабочая. Во-первых, хотя оптимальное действие одно, как правило, хороших действий обычно несколько. Во-вторых, по-хорошему здесь нужно оценивать что-то типа accuracy@k, потому что мы хотим скорее ранжирование действий по их оптимальности. В общем случае это вообще сложно придумать, потому что непонятно какой брать k. В нашем случае он 5 (и то неправда, если `uniform_set`), а в реальности какой брать - одному богу известно

2. Mean regret. Это отклонение от оптимального действия, измеряемое в reward. Уже получше, во всяком случае можно разных бандитов по этой метрике сравнивать. Даже минимизировать можно, в оригинальном `DecisionTransformer` так и делали

3. Mean reward. На мой взгляд лучший вариант. Средний reward у случайного алгоритма должен быть около 0.5, я так генерировал все выборки. Если он выше этого значения (или ниже), значит мы чему-то научились, потому что какое-никакое ранжирование есть. Не оптимальное, ну да никто не идеален

In [89]:
import polars as pl

wandb_df = pd.read_csv(Path(DATASET_DIR, "wandb_runs.csv")).drop(
    ["Notes", "User", "Sweep"], axis=1
)
wandb_df = pl.from_pandas(wandb_df[wandb_df != ""].dropna())

In [91]:
wandb_df = wandb_df.rename(
    {
        "valid_accuracy/dataloader_idx_0": "even_set__accuracy",
        "valid_accuracy/dataloader_idx_1": "uniform_set__accuracy",
        "valid_mean_regret/dataloader_idx_0": "even_set__mean_regret",
        "valid_mean_regret/dataloader_idx_1": "uniform_set__mean_regret",
        "valid_mean_reward/dataloader_idx_0": "even_set__mean_reward",
        "valid_mean_reward/dataloader_idx_1": "uniform_set__mean_reward",
    }
)

Сперва посмотрим, а чего мы вообще добились, обучились ли хоть чему-то или нет. У нас тут вообще парето фронт, потому что датасета два. Ну давайте уже какую-нибудь грубую эвристику, типа усреднения возьмём, время поджимает. И оценивать будем по средней награде

In [92]:
valid_reward_cols = ["even_set__mean_reward", "uniform_set__mean_reward"]

In [113]:
score_sorted = wandb_df.with_columns(
    mean_score=(
        pl.col("even_set__mean_reward") + pl.col("even_set__mean_reward")
    )
    / 2
).sort("mean_score")

score_sorted.tail(1).select(valid_reward_cols + ["train_mean_reward"]).plot.bar(
    title="Best reward distribution"
)

:Bars   [index,Variable]   (value)

In [116]:
score_sorted.tail(1)

Name,State,Tags,Created,Runtime,arm_distributions.even,arm_distributions.odd,arm_distributions.uniform,attention_dropout,emb_strategy,embedding_dim,embedding_dropout,episode_len,feedforward_dim,max_action,num_actions,num_heads,num_layers,num_states,optimizer,optimizer_kwargs.betas,optimizer_kwargs.lr,residual_dropout,scheduler,scheduler_kwargs.T_0,scheduler_kwargs.eta_min,seq_len,epoch,train_accuracy,train_loss,train_mean_regret,train_mean_reward,trainer/global_step,even_set__accuracy,uniform_set__accuracy,valid_loss/dataloader_idx_0,valid_loss/dataloader_idx_1,even_set__mean_regret,uniform_set__mean_regret,even_set__mean_reward,uniform_set__mean_reward,mean_score
str,str,str,str,i64,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""BanditDT_lr=5e-06_layers=8_seq…","""finished""","""bandit""","""2024-08-13T22:44:29.000Z""",1492,"""[0.715158224105835,0.253837019…","""[0.26788556575775146,0.7275376…","""[0.5012867450714111,0.49775734…",0.1,"""stack""",64.0,0.1,20000.0,256.0,1.0,10.0,4.0,8.0,0.0,"""torch.optim.adam.Adam""","""[0.9,0.99]""",0.000005,0.1,"""torch.optim.lr_scheduler.Cosin…",1000.0,0.000002,64.0,25.0,0.782969,0.589955,0.082291,0.735786,19999.0,0.0054,0.6788,2.784668,1.696948,0.451264,0.203026,0.727138,0.692793,0.727138


По картинке видно, что какой-то из конфигураций моделей удаётся добится сопоставимого реворда на всех датасетах, из чего я делаю вывод, что статья рабочая и всего достичь можно. К сожалению, не сохранял число параметров, но их там не 10М, это я могу гарантировать. Но с другой стороны и reward не близок к 1, он здесь в районе 0.7, ещё можно учить и учить, так что нужно больше экспериментов

In [117]:
wandb_df.groupby("num_heads").agg(
    pl.col("even_set__mean_reward", "uniform_set__mean_reward").mean()
)

num_heads,even_set__mean_reward,uniform_set__mean_reward
f64,f64,f64
4.0,0.551771,0.637677


По головам собрать статистику не успел :(

In [122]:
wandb_df.groupby("num_layers").agg(
    pl.col("even_set__mean_reward", "uniform_set__mean_reward").mean()
).sort("num_layers").plot.line(
    x="num_layers",
    y=valid_reward_cols,
    title="Reward depednding on layer_number",
)

:NdOverlay   [Variable]
   :Curve   [num_layers]   (value)

По тому, что вышло у меньше, число слоёв сильного влияния не оказало, но они вляют не одни, а вкупе с размерностью эмбеддинга всё-таки. Так что до совсем большого числа параметров, типа 10М я не добрался

In [125]:
wandb_df.groupby("embedding_dim").agg(pl.col(valid_reward_cols).mean()).sort(
    "embedding_dim"
).plot.line(
    x="embedding_dim",
    y=valid_reward_cols,
    title="Reward depednding on embedding_dim",
)

:NdOverlay   [Variable]
   :Curve   [embedding_dim]   (value)

По размерности эмбеддингов я тоже ничего не могу сказать, потому что фактически перебрал лишь два параметра. Я бы предположил, что чем она больше, и чем больше у нас слоёв, тем будет выше reward, за счёт большего числа параметров. Если мы конечно не переобучимся, это наблюдалось сплошь и рядом во время обучения

In [128]:
wandb_df.groupby("optimizer_kwargs.lr").agg(
    pl.col(valid_reward_cols).mean()
).sort("optimizer_kwargs.lr").plot.line(
    x="optimizer_kwargs.lr",
    y=valid_reward_cols,
    title="Reward depednding on lr",
)

:NdOverlay   [Variable]
   :Curve   [optimizer_kwargs.lr]   (value)

С оптимайзером всё довольно однозначно. ПО всей видимости стоит его оставить большим, как он и есть. В статье не указывали, к сожалению

In [130]:
wandb_df.groupby("seq_len").agg(pl.col(valid_reward_cols).mean()).sort(
    "seq_len"
).plot.line(
    x="seq_len",
    y=valid_reward_cols,
    title="Reward depednding on seq_len",
)

:NdOverlay   [Variable]
   :Curve   [seq_len]   (value)

Длину последовательности тоже не оценил, но очень надо

Ну и видимо всё. С аналитикой, к сожалению, не задалось, потому что собрал слишком мало, по не зависящим от меня причинам. Как же я не люблю колаб(((